# DPO Fine-tuning: Llama-3.1-8B for Long-Context Faithfulness

**Goal:** Fine-tune `Meta-Llama-3.1-8B-Instruct` via DPO to improve:
- **Faithfulness under distraction** — ignore irrelevant context chunks
- **False-premise rejection** — detect and correct wrong assumptions in questions
- **Lost-in-the-middle robustness** — retrieve relevant info regardless of its position

**Method:** LoRA (r=16) + DPO via TRL, using Unsloth for 4-bit quantization.

**Runtime:** ~2.5–3h on Kaggle T4×2 GPU.

**Dataset:** `dpo_train.jsonl` / `dpo_eval.jsonl` — generated locally by `build_dpo.py`.

---
**Before running:** Upload `dpo_train.jsonl` and `dpo_eval.jsonl` via Kaggle Datasets or the notebook file upload panel.

In [ ]:
# Cell 2: Install dependencies
# Unsloth installs torch-compatible versions of transformers/peft/trl automatically.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install trl==0.12.2 datasets==3.1.0 accelerate bitsandbytes==0.44.1 --quiet
print('Install complete.')

In [ ]:
# Cell 3: Imports and setup
import os
import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

import torch
from datasets import Dataset
from trl import DPOConfig, DPOTrainer
from unsloth import FastLanguageModel

# Hugging Face token (add HF_TOKEN to Kaggle Secrets)
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Cell 4: Load base model via Unsloth (4-bit quantized)
MAX_SEQ_LENGTH = 4096
MODEL_ID = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
)
print(f'Model loaded: {MODEL_ID}')
print(f'Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Cell 5: Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'LoRA attached. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 6: Load dataset
# Upload dpo_train.jsonl and dpo_eval.jsonl before running this cell.
# Expected location: /kaggle/input/<dataset-name>/ or /kaggle/working/

def load_jsonl(path: str) -> list:
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

# Adjust paths if you uploaded to a Kaggle Dataset
TRAIN_PATH = '/kaggle/working/dpo_train.jsonl'
EVAL_PATH  = '/kaggle/working/dpo_eval.jsonl'

train_raw = load_jsonl(TRAIN_PATH)
eval_raw  = load_jsonl(EVAL_PATH)
print(f'Train: {len(train_raw)} pairs | Eval: {len(eval_raw)} pairs')

# Type breakdown
from collections import Counter
train_types = Counter(d['meta']['type'] for d in train_raw)
print(f'Train type distribution: {dict(train_types)}')

In [ ]:
# Cell 7: Format dataset with Llama-3.1 chat template
# DPOTrainer expects columns: prompt, chosen, rejected (plain strings or chat lists).
# We pass plain strings — TRL will apply the tokenizer's chat template internally.

def to_hf_dataset(records: list) -> Dataset:
    return Dataset.from_dict({
        'prompt':   [r['prompt']   for r in records],
        'chosen':   [r['chosen']   for r in records],
        'rejected': [r['rejected'] for r in records],
    })

train_ds = to_hf_dataset(train_raw)
eval_ds  = to_hf_dataset(eval_raw)

print(f'train_ds: {train_ds}')
print(f'eval_ds:  {eval_ds}')
print('\nSample prompt (first 300 chars):')
print(train_ds[0]['prompt'][:300])

In [ ]:
# Cell 8: DPO training configuration
OUTPUT_DIR = '/kaggle/working/llama31-8b-lora-adapter'

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch = 8
    learning_rate=5e-6,              # conservative for 8B
    beta=0.1,                        # DPO regularization
    warmup_ratio=0.1,
    optim='adamw_8bit',
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=3600,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,   # Unsloth handles reference model internally
    args=dpo_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)
print('DPOTrainer initialized.')

In [ ]:
# Cell 9: Train
# Expected runtime: ~2.5-3h on T4 x2 with ~500 train pairs.
train_result = trainer.train()
print('Training complete.')
print(train_result.metrics)

In [ ]:
# Cell 10: Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')
print('Files:', list(Path(OUTPUT_DIR).iterdir()))

# --- DOWNLOAD INSTRUCTIONS ---
# After the notebook finishes:
#   Kaggle Notebook → Output tab → Download "llama31-8b-lora-adapter/" folder
#   Extract to: 02-hallucination/mitigation/checkpoints/llama31-8b-lora-adapter/

## Cell 11 Setup: Add GROQ_API_KEY Secret

Before running Cell 11, add your Groq API key to Kaggle Secrets:
1. Sidebar → **Add-ons** → **Secrets**
2. Add secret named `GROQ_API_KEY` with your key value
3. Attach it to this notebook session

Cell 11 uses `llama-3.1-8b-instant` via Groq API as the **"Before DPO"** baseline
(same base model family, no adapter), and the local DPO adapter as **"After DPO"**.

In [ ]:
# Cell 11: Before / After DPO inference comparison
# "Before" = Groq llama-3.1-8b-instant (same base family, no DPO adapter)
# "After"  = local Llama-3.1-8B + LoRA adapter just trained

!pip install groq==0.11.0 --quiet

import textwrap
from groq import Groq
from unsloth import FastLanguageModel

secrets = UserSecretsClient()
groq_client = Groq(api_key=secrets.get_secret('GROQ_API_KEY'))

# Load DPO adapter for "After" inference
FastLanguageModel.for_inference(model)

TEST_PROMPTS = [
    # 2 grounded questions (answer must be in context)
    {
        'label': 'Grounded-1',
        'context': eval_raw[0]['prompt'],
        'question': eval_raw[0]['prompt'].split('Câu hỏi:')[-1].strip(),
    },
    {
        'label': 'Grounded-2',
        'context': eval_raw[1]['prompt'],
        'question': eval_raw[1]['prompt'].split('Câu hỏi:')[-1].strip(),
    },
    # 2 false-premise questions (model should reject/correct)
    {
        'label': 'FalsePremise-1',
        'context': next(r for r in eval_raw if r['meta']['type'] == 'premise')['prompt'],
        'question': next(r for r in eval_raw if r['meta']['type'] == 'premise')['prompt'].split('Câu hỏi:')[-1].strip(),
    },
    {
        'label': 'FalsePremise-2',
        'context': [r for r in eval_raw if r['meta']['type'] == 'premise'][1]['prompt'],
        'question': [r for r in eval_raw if r['meta']['type'] == 'premise'][1]['prompt'].split('Câu hỏi:')[-1].strip(),
    },
    # 1 distractor question
    {
        'label': 'Distractor-1',
        'context': next(r for r in eval_raw if r['meta']['type'] == 'distractor')['prompt'],
        'question': next(r for r in eval_raw if r['meta']['type'] == 'distractor')['prompt'].split('Câu hỏi:')[-1].strip(),
    },
]

SYSTEM_GROUNDED = (
    'Bạn là trợ lý AI chỉ trả lời dựa trên tài liệu được cung cấp. '
    'Nếu thông tin không có trong tài liệu, nói "Thông tin không có trong tài liệu."'
)

def generate_before(prompt_text: str) -> str:
    resp = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': SYSTEM_GROUNDED},
            {'role': 'user', 'content': prompt_text},
        ],
        max_tokens=512,
        temperature=0.1,
    )
    return resp.choices[0].message.content.strip()

def generate_after(prompt_text: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_GROUNDED},
        {'role': 'user', 'content': prompt_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
        )
    generated = outputs[0][inputs.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Run comparison
print('Running Before/After comparison on 5 test prompts...\n')
rows = []
for i, item in enumerate(TEST_PROMPTS):
    print(f'[{i+1}/5] {item["label"]}...')
    before = generate_before(item['context'])
    after  = generate_after(item['context'])
    rows.append({'label': item['label'], 'before': before, 'after': after})

# Print markdown table
wrap = lambda s: textwrap.fill(s, width=60).replace('\n', ' ')
print('\n## Before / After DPO Comparison\n')
print(f'| # | Label | Before DPO (Groq 8B-instant) | After DPO (LoRA adapter) |')
print(f'|---|-------|------------------------------|--------------------------|')
for i, row in enumerate(rows, 1):
    print(f'| {i} | {row["label"]} | {wrap(row["before"])[:120]}... | {wrap(row["after"])[:120]}... |')